In [3]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [19]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="readerbench/echo", revision="refs/convert/parquet",
    repo_type="dataset", local_dir="./echo", allow_patterns="*/*.parquet")

Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 2769.02it/s]


'/home/ubuntu/echo'

In [22]:
files = glob('echo/default/*/*.parquet')
len(files)

22

In [25]:
df = pd.read_parquet(files[0])
df

,audio,path,text,speaker,age,gender
0,{'bytes': b'RIFF\xc6\x9c\x01\x00WAVEfmt \x10\x...,audio/ed/edfae9b3ef14fb8dbb1e5a93ff683c8f.wav,de mâncare îmi aduce popa gavrilă,000057,,F
1,{'bytes': b'RIFF\xc6B\x01\x00WAVEfmt \x10\x00\...,audio/50/50004524e1db5381808db4eb8641c2aa.wav,și asta îl îndârji și mai grozav,000041,,F
2,{'bytes': b'RIFF\xc6\xc7\x03\x00WAVEfmt \x10\x...,audio/92/920fffd7e4ca5b83ca95749cac4bc222.wav,această podgorie are peste o sută treizeci de ...,000124,,F
3,{'bytes': b'RIFFFX\x02\x00WAVEfmt \x10\x00\x00...,audio/51/51fa96cbfe57669d2837322473456b69.wav,faptul că nu-mi găseam liniștea indiferent ce ...,000124,,F
4,{'bytes': b'RIFFF\xdf\x02\x00WAVEfmt \x10\x00\...,audio/67/67d68ca59b89d18ba2bba83c2cecbc20.wav,este posibil că aceste populații să fie subspe...,000074,,F
...,...,...,...,...,...,...
2619,{'bytes': b'RIFF\xc6\x05\x02\x00WAVEfmt \x10\x...,audio/6a/6a3a92279474451b659255e90bea32bb.wav,cel de-al cincilea album de studio intitulat j...,000181,,
2620,{'bytes': b'RIFF\xc6\xc5\x05\x00WAVEfmt \x10\x...,audio/de/de43ae853c1074bb51e6b8a5042abb0b.wav,unu cererea persoanei care dorește să devină a...,000154,,
2621,{'bytes': b'RIFFFb\x07\x00WAVEfmt \x10\x00\x00...,audio/57/57b6bc8f7331d635c093c49bce17d1dd.wav,șapte obligația prevăzută la aliniatul unu est...,000124,,F
2622,{'bytes': b'RIFF\xc6@\x03\x00WAVEfmt \x10\x00\...,audio/0e/0ef468ec4e4b7b42172a0aa89b40d684.wav,studenții care apelează la alte persoane să le...,000113,,M


In [26]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker'].iloc[i]}"
            })
        
    return data

In [27]:
data = multiprocessing(files, loop, cores = 5)

100%|██████████| 2812/2812 [03:03<00:00, 15.30it/s]


In [28]:
len(data)

51727

In [29]:
with open('echo.json', 'w') as fopen:
    json.dump(data, fopen)

In [30]:
audio_files = [d['audio_filename'] for d in data]

with open('echo-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [31]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'echo_audio/echo-default-partial-train-0005_0.mp3',
 'text': 'de mâncare îmi aduce popa gavrilă',
 'speaker': 'echo_audio_000057'}

In [32]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'echo')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 42.55ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 2.93MB / 2.93MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 2.93MB / 2.93MB,  0.00B/s  
New Data Upload: 100%|██████████| 2.93MB / 2.93MB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.64 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/0b9322c62fcf9c0f03564b76ac122cbd1d2c76c8', commit_message='Upload dataset', commit_description='', oid='0b9322c62fcf9c0f03564b76ac122cbd1d2c76c8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [5]:
# !zip -rq echo_audio.zip echo_audio

In [4]:
# !hf upload malaysia-ai/Multilingual-TTS echo_audio.zip --repo-type=dataset

In [8]:
# !zip -rq echo_audio_neucodec.zip echo_audio_neucodec

In [9]:
# !hf upload malaysia-ai/Multilingual-TTS echo_audio_neucodec.zip --repo-type=dataset